# 9 — Generate text

**Before:** notebook **8** (training). Loads `checkpoints/tiny_gpt.pt` if you trained in nb 8.

**This notebook:** `generate()` — inference loop from llm.c / nanoGPT.

**Learning objectives**

- Load a checkpoint from notebook 8 when available.
- Generate text with `generate()` (temperature, top-k).
- Compare untrained vs trained sample quality.
- Relate sampling to inference in llm.c / nanoGPT.


In [ ]:
# --- Setup: find repo root (llm-c-from-scratch or Cursor workbook) ---
import sys
from pathlib import Path


def find_llm_root() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "llmc" / "__init__.py").is_file():
            return base
        nested = base / "llm-c-from-scratch"
        if (nested / "llmc" / "__init__.py").is_file():
            return nested
    return Path.cwd()


ROOT = find_llm_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from llmc.notebook_utils import data_path, checkpoint_path

DATA = data_path(ROOT)
CHECKPOINT = checkpoint_path(ROOT)
print("ROOT", ROOT.resolve())
print("data", "OK" if DATA.is_file() else "missing")


In [ ]:
import torch
from llmc.data import CharTokenizer, load_text
from llmc.model import GPT, GPTConfig
from llmc.sample import generate

text = load_text(DATA)
tok = CharTokenizer.from_text(text)
config = GPTConfig.tiny(vocab_size=tok.vocab_size, block_size=64)
model = GPT(config)
device = "cuda" if torch.cuda.is_available() else "cpu"

if CHECKPOINT.is_file():
    ckpt = torch.load(CHECKPOINT, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model"])
    print("loaded checkpoint", CHECKPOINT)
else:
    print("no checkpoint — untrained model (gibberish). Run notebook 8 first.")

model = model.to(device)
prompt = "ROMEO:"
ctx = tok.encode_tensor(prompt, device=device).unsqueeze(0)
out = generate(model, ctx, max_new_tokens=200, temperature=0.8, top_k=40)
print(tok.decode(out.squeeze().tolist()))
